In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Introduction

This notebook contains code for a micro-Small Language Model (< 2 Million parameters) consisting of a Language Model with 1-2 transformer blocks and multi-headed, causal self-attention. This model is pretrained on the TinyStories dataset for the next-token prediction task. The pretrained model is then repeatedly input generated text to generate a short story.

# Steps:
1. **Data Exploration**
2. **Data Preprocessing**
     1. Tokenization
     2. Saving data to memory
     3. Input-target pairing
3. **Building Model Architecture**
    1. Projection to Token Embedding Space
    2. Projection to Positional Embedding Space (for later; unclear on this)
    3. Attention
    4. Feed-forward Network/ Multi-layer Perceptrons
    5. Layer Normalization
    6. Transformer
    7. Small Language Model (contains embedding projection, transformer blocks, LM Head, etc.)
4. **Pre-training**
    1. Defining cross-entropy loss function
    2. Updating parameters using backpropagation
    3. Pre-training using training data
    4. Validate model on validation dataset
    5. Visualize and analyze training and validation losses
    6. Hyperparameter tuning and repeat training and validation
5. **Short Story Generation**
6. **Explore and Evaluate Generated Output**


# Step 1: Data Exploration

In [ ]:
if not os.path.exists('./data'):
    os.makedirs('./data')
    
kagglehub.dataset_download('thedevastator/tinystories-narrative-classification',
                           path = 'train.csv', output_dir = './data')
kagglehub.dataset_download('thedevastator/tinystories-narrative-classification',
                           path = 'validation.csv', output_dir = './data')

In [ ]:
train = pd.read_csv('/kaggle/input/datasets/thedevastator/tinystories-narrative-classification/train.csv')
val = pd.read_csv('/kaggle/input/datasets/thedevastator/tinystories-narrative-classification/validation.csv')

In [ ]:
pd.options.display.max_rows = 500

print("Summary of training dataset:")
print(train.describe)

In [ ]:
print("Summary of validation dataset:")
print(val.describe)

In [ ]:
print('First story in training set:')
print(train.iloc[0]['text'])

## Select sample of 10 stories for initial run 

In [ ]:
print('2 more stories from training set and validation set each:')
train_sample = train.sample(10, random_state = 1)
val_sample = val.sample(2, random_state = 1)

print(train_sample.iloc[0]['text'])
print('-------------X-------------')
print(train_sample.iloc[1]['text'])
print('-------------X-------------')
print(val_sample.iloc[0]['text'])
print('-------------X-------------')
print(val_sample.iloc[1]['text'])

## Tokenize Data and Create Chunks of Input-Target Pairs

Firstly, we must break the natural language text into units called **tokens** and assign a numerical Id to each called a **token ID**. The list of containing all the units in the language, be they words, sub-words, or byte-pairs, indexed by their corresponding token IDs is called the **vocabulary**. For this task, we will use a pre-existing GPT2 tokenizer to perform **Byte-Pair Encoding (BPE)**.

For the next token prediction task, a **sliding window** is defined, such that given the tokens in the window, i.e. the **context**, the next token in the sequence is considered to be the expected output. The max number of tokens that can be in the window is the **context size**. Tokenized data is grouped into these input (the tokens in the window) and target (the next expected token) pairs. These groups are called **chunks**.

In [ ]:
!pip install git+https://github.com/huggingface/transformers

In [ ]:
import torch.nn.functional as F
import torch
from transformers import GPT2Tokenizer, BatchEncoding

In [ ]:
'''
tokenizer = GPT2Tokenizer.from_pretrained('openai-community/gpt2', add_eos_token = True)
print('Vocabulary size = ', tokenizer.vocab_size)

trainTokens = tokenizer(train_sample['text'].tolist())
valTokens = tokenizer(val_sample['text'].tolist())

print(trainTokens[0])
print('---------------X--------------')
print(valTokens[0])
print('---------------X--------------')
print(trainTokens['input_ids'][0])
print('---------------X--------------')
print(len(trainTokens['input_ids']))
print('---------------X--------------')
print(tokenizer.decode(trainTokens['input_ids'][0]))
'''

In [ ]:
class Dataset:
    def __init__(self, rawData, contextSize, split):
        self.split = split
        tokenizer = GPT2Tokenizer.from_pretrained('openai-community/gpt2', add_eos_token = True)
        tokenData = tokenizer(rawData)
        tokenIds = []
        for document in tokenData['input_ids']:
            tokenIds.extend(document)
        # Chunking tokens based on input and target
        self.inputTokens = torch.zeros((len(tokenIds) - contextSize,contextSize))
        self.targetTokens = torch.zeros((len(tokenIds) - contextSize,contextSize))
        for i in range(len(tokenIds) - contextSize + 1):
            inputTokens[i,:] = tokenIds[i:contextSize]
            targetTokens[i,:] = tokenIds[i+1:contextSize+1]
            i = i + contextSize

    def __len__(self):
        return inputTokens.size(dim=0)

    def __getitem__(self, idx):
        return inputTokens[idx,:], targetTokens[idx,:]

In [ ]:
'''
def io_pairing(data):
    inputTokens, outputTokens = [], []
    # Leave out the last token;it is never input to the model
    for document in data:    
        inputTokens.extend(document[0:len(document) - 1])
        outputTokens.extend(document[1:len(document)])

    return inputTokens, outputTokens
    '''

In [ ]:
'''
inputTokens, outputTokens = io_pairing(trainTokens['input_ids'])
print(inputTokens[0:4])
print(outputTokens[0:4])
print(inputTokens[len(inputTokens) - 4:len(inputTokens)])
print(outputTokens[len(outputTokens) - 4:len(outputTokens)])
'''

In [ ]:
# inputValTokens, outputValTokens = io_pairing(valTokens['input_ids'])

### Batch Data to Prepare it for the Embedding Layer of the Model

In [ ]:
# an iterable class to batch data in dataset and iterated through the batches
class DatasetLoader:
    def __init__(self, dataset, batchSize, contextSize):
        dataLen = dataset.__len__()
        assert dataLen // batchSize
        # Batching data from Dataset object
        self.numOfBatches = dataLen / batchSize
        self.currentBatch = 0
        self.dataset = dataset

    def __iter__(self):
        return self

    def __next__(self):
        inputData = torch.zeros((batchSize, contextSize))
        targetData = torch.zeros((batchSize, contextSize))
        # get data for current batch and collate it
        index = currentBatch * batchSize
        for i in range(batchSize):
            index = index + i
            inputData[index,:], outputData[index,:]= self.dataset[index] # syntactical sugar for __getitem__()
        self.currentBatch = self.currentBatch + 1
        return inputData, outputData

## Project to Embedding Space

A set of numbers, such as these **tokens**, carry no information whatsoever to a computer. To imbue these tokens with some **semantic meaning**, we project them to a higher order **embedding space**. Thus, each token becomes a vector of *n* dimensions, where *n* is the dimension of the embedding space.

These weights, first initialized randomly, will be learned by the Language Model during training.

In [ ]:
embedMatrix = torch.rand(vocabSize, emdbDim)
def embedding_layer(vocabSize, embdDim, inputTokens):
    return torch.matmul(F.one_hot(inputTokens, num_classes = vocabSize), embedMatrix.unsqueeze(2))

In [ ]:
'''
embedMatrix, embedInput, embedOutput = generate_embd(tokenizer.vocab_size, 
                                                     128, 
                                                     torch.from_numpy(inputTokens), 
                                                     torch.from_numpy(outputTokens))
embedMatrix.shape
embedInput.shape
embedOutput.shape
'''